# 🔬 Session 02 — Unsupervised Advanced Clustering Algorithms

**Covers**: TL3 — Clustering Techniques, Dimensionality Reduction

**Duration**: 2 Hours | **Level**: 🟢🟡🔴 All Levels

---

## What You'll Learn
1. K-Means clustering and the elbow/silhouette methods
2. DBSCAN for non-spherical clusters and anomaly detection
3. Hierarchical clustering with dendrograms
4. Gaussian Mixture Models (soft clustering)
5. PCA, t-SNE, UMAP for dimensionality reduction
6. Choosing the right clustering algorithm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.datasets import make_blobs, make_moons, make_circles, load_digits, load_iris
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
print('✅ Libraries loaded!')

---
## Part 1: K-Means Customer Segmentation

In [ ]:
# Generate realistic customer data
np.random.seed(42)
segments = [
    {'n': 60, 'income': (25, 8), 'spending': (15, 8)},
    {'n': 40, 'income': (30, 10), 'spending': (75, 10)},
    {'n': 60, 'income': (55, 12), 'spending': (50, 12)},
    {'n': 40, 'income': (85, 8), 'spending': (25, 8)},
    {'n': 40, 'income': (90, 8), 'spending': (80, 8)},
]

incomes, spendings = [], []
for s in segments:
    incomes.extend(np.random.normal(*s['income'], s['n']))
    spendings.extend(np.random.normal(*s['spending'], s['n']))

customers = pd.DataFrame({
    'Income_K': np.clip(incomes, 10, 130),
    'Spending_Score': np.clip(spendings, 1, 100)
})

X = StandardScaler().fit_transform(customers)

# Find optimal K
K_range = range(2, 11)
inertias, sil_scores = [], []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels))

optimal_k = list(K_range)[np.argmax(sil_scores)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('K-Means Customer Segmentation', fontsize=16, fontweight='bold')

axes[0].plot(K_range, inertias, 'bo-'); axes[0].set_title('Elbow Method')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')

axes[1].bar(K_range, sil_scores, color='#2ecc71')
axes[1].set_title(f'Silhouette Score (Best K={optimal_k})')

km_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
customers['Cluster'] = km_final.fit_predict(X)
scatter = axes[2].scatter(customers['Income_K'], customers['Spending_Score'],
                          c=customers['Cluster'], cmap='viridis', alpha=0.6, s=40)
axes[2].set_title(f'{optimal_k} Customer Segments'); axes[2].set_xlabel('Income ($K)')
axes[2].set_ylabel('Spending Score')

plt.tight_layout()
plt.show()

# Segment profiles
for c in range(optimal_k):
    seg = customers[customers['Cluster']==c]
    print(f'Cluster {c}: {len(seg)} customers | Income: ${seg["Income_K"].mean():.0f}K | Spending: {seg["Spending_Score"].mean():.0f}')

---
## Part 2: DBSCAN vs K-Means

In [ ]:
# DBSCAN shines with non-spherical clusters
X_moons, _ = make_moons(n_samples=300, noise=0.05, random_state=42)
X_circles, _ = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('K-Means vs DBSCAN on Different Shapes', fontsize=16, fontweight='bold')

for row, (X_data, name) in enumerate([(X_moons, 'Moons'), (X_circles, 'Circles')]):
    Xs = StandardScaler().fit_transform(X_data)
    
    axes[row,0].scatter(X_data[:,0], X_data[:,1], c='gray', s=20, alpha=0.6)
    axes[row,0].set_title(f'{name}: Original')
    
    km_labels = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(Xs)
    axes[row,1].scatter(X_data[:,0], X_data[:,1], c=km_labels, cmap='viridis', s=20)
    axes[row,1].set_title(f'K-Means ❌')
    
    db_labels = DBSCAN(eps=0.3, min_samples=5).fit_predict(Xs)
    noise = (db_labels==-1).sum()
    axes[row,2].scatter(X_data[:,0], X_data[:,1], c=db_labels, cmap='viridis', s=20)
    axes[row,2].set_title(f'DBSCAN ✅ ({noise} noise pts)')

plt.tight_layout()
plt.show()
print('✅ DBSCAN handles non-spherical clusters where K-Means fails!')

---
## Part 3: Hierarchical Clustering

In [ ]:
iris = load_iris()
np.random.seed(42)
idx = np.random.choice(len(iris.data), 30, replace=False)
X_sample = iris.data[idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Hierarchical Clustering', fontsize=16, fontweight='bold')

linkage_matrix = linkage(X_sample, method='ward')
dendrogram(linkage_matrix, ax=axes[0], truncate_mode='level', p=5)
axes[0].set_title('Dendrogram (Ward Linkage)')
axes[0].axhline(y=8, color='red', linestyle='--', label='Cut → 3 clusters')
axes[0].legend()

labels = AgglomerativeClustering(n_clusters=3).fit_predict(iris.data)
axes[1].scatter(iris.data[:,0], iris.data[:,2], c=labels, cmap='viridis', alpha=0.6, s=40)
axes[1].set_xlabel(iris.feature_names[0]); axes[1].set_ylabel(iris.feature_names[2])
axes[1].set_title('3 Clusters on Iris Data')

plt.tight_layout()
plt.show()

---
## Part 4: Gaussian Mixture Models

In [ ]:
X_gmm, _ = make_blobs(n_samples=400, centers=3, cluster_std=[1.5, 1.0, 0.5], random_state=42)

gmm = GaussianMixture(n_components=3, random_state=42)
gmm_labels = gmm.fit_predict(X_gmm)
probs = gmm.predict_proba(X_gmm)

bics = [GaussianMixture(n_components=n, random_state=42).fit(X_gmm).bic(X_gmm) for n in range(1,8)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Gaussian Mixture Models', fontsize=16, fontweight='bold')

axes[0].scatter(X_gmm[:,0], X_gmm[:,1], c=gmm_labels, cmap='viridis', alpha=0.6, s=20)
axes[0].set_title('GMM Hard Assignment')

sc = axes[1].scatter(X_gmm[:,0], X_gmm[:,1], c=probs[:,0], cmap='RdYlGn', alpha=0.6, s=20)
plt.colorbar(sc, ax=axes[1], label='P(Cluster 0)')
axes[1].set_title('Soft Assignment (Probabilities)')

axes[2].plot(range(1,8), bics, 'bo-')
axes[2].set_title(f'BIC (Optimal: {np.argmin(bics)+1} components)')
axes[2].set_xlabel('Components'); axes[2].set_ylabel('BIC')

plt.tight_layout()
plt.show()

---
## Part 5: Dimensionality Reduction (PCA & t-SNE)

In [ ]:
digits = load_digits()
X_digits = StandardScaler().fit_transform(digits.data)
y_digits = digits.target

print(f'Original: {X_digits.shape[1]} dimensions (8x8 pixel images)')

# PCA to 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_digits)
print(f'PCA 2D: {pca.explained_variance_ratio_.sum():.1%} variance retained')

# PCA 95% variance
pca_95 = PCA(n_components=0.95).fit(X_digits)
print(f'Components for 95% variance: {pca_95.n_components_}')

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_digits)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('64D → 2D: Dimensionality Reduction', fontsize=16, fontweight='bold')

sc1 = axes[0].scatter(X_pca[:,0], X_pca[:,1], c=y_digits, cmap='tab10', s=5, alpha=0.5)
axes[0].set_title(f'PCA ({pca.explained_variance_ratio_.sum():.0%} variance)')

axes[1].scatter(X_tsne[:,0], X_tsne[:,1], c=y_digits, cmap='tab10', s=5, alpha=0.5)
axes[1].set_title('t-SNE (better separation)')

cumvar = np.cumsum(PCA().fit(X_digits).explained_variance_ratio_)
axes[2].plot(cumvar)
axes[2].axhline(0.95, color='r', linestyle='--', label='95%')
axes[2].set_title('Cumulative Explained Variance')
axes[2].set_xlabel('Components'); axes[2].legend()

plt.colorbar(sc1, ax=axes[0], label='Digit')
plt.tight_layout()
plt.show()

---
## ✅ Session 02 Complete!

**Key Takeaways:**
- K-Means for spherical clusters (use elbow + silhouette to find K)
- DBSCAN for any shape + outlier detection (no K needed)
- Hierarchical for dendrogram visualization
- GMM for soft/probabilistic cluster assignments
- PCA for feature reduction, t-SNE/UMAP for visualization

**Next:** Session 03 — Markov Decision Process & Reinforcement Learning